# 01 · Generate structures for MLIP fine-tuning

## Objective

This notebook generates and selects structures to label using DFT for a fine-tuning dataset. It uses `DatasetGenerationWorkflowMaker` with demonstration settings.

The workflow has these steps:
1. Generate initial LiNbO3 and SiO2 structures with PyXtal.
2. Relax the structures with the foundational MatQ model.
3. Create variants. Change atom positions and cell dimensions. Filter the variants.
4. Select structures with DIRECT sampling.

This notebook generates structures only. Notebook 02 shows DFT labelling and Notebook 03 uses labelled structures to fine-tune the foundational model. The small settings show the procedure. They do not produce a validated training dataset.

## Prerequisites
Before running the notebook, ensure that the following dependencies are met,

1. Install the notebook and MLIP dependencies:

   ```bash
   uv sync --group notebooks --group mlip
   ```

2. Make sure that `mlip_models_dir` is set in [programs] in `crystal.toml`. See README for more details.

## 1. Import the workflow tools

Import the settings, model reference, workflow maker, and storage tools. The workflow runs on this computer. No run flag is used.

In [ ]:
from jobflow.core.store import JobStore
from jobflow.managers.local import run_locally
from maggma.stores import MemoryStore

from crystal import CRYSTAL_ROOT
from crystal.constants.mlip import MLIP_RELAX_TASK_DOCUMENT_BLOB_STORE_NAME
from crystal.constants.structure_generators import STRUCTURE_BLOB_STORE_NAME
from crystal.schemas.formula_factories import FixedSetFormulaFactorySettings
from crystal.schemas.initial_structure_generation import (
    InitialStructureGenerationSettings,
)
from crystal.schemas.mlip import MlipRelaxMakerSettings, MlipRelaxSimulationSettings
from crystal.schemas.mlip_model_ref import MlipModelRef, QetModelRegistry
from crystal.schemas.structure_generators import (
    BirchClusteringSettings,
    ClusterSamplingSettings,
    DatasetGenerationWorkflowSettings,
    DatasetGenerationWorkflowTaskDocument,
    DirectSamplingMakerSettings,
    DirectSamplingSettings,
    InitialStructureGenerationMakerSettings,
    RattleAtomPositionsSettings,
    StretchCellSettings,
    StructureAugmentationAndFilteringMakerSettings,
    StructureAugmentationAndFilteringSettings,
)
from crystal.structure.generators.dataset_generation_workflow_maker import (
    DatasetGenerationWorkflowMaker,
)


## 2. Set the generation parameters

This uses `DatasetGenerationWorkflowSettings` to configure the workflow. Here, we request four initial structures across the two formulas. Use space group 1 and fixed random seeds and limit each relaxation to two steps for faster execution. These settings reduce the calculation time, but do not ensure convergence.

Create variants with small changes to atom positions and cell dimensions. Use one cluster for DIRECT sampling and request four samples from that cluster. Filtering and selection can change the final structure count.

Set the run directory for calculation files. The workflow results will remain in memory.

In [ ]:
NUMBER_OF_INITIAL_STRUCTURES = 4
MAX_RELAX_STEPS = 2
RUN_DIRECTORY = CRYSTAL_ROOT / "examples/results/generation-runs"

model = MlipModelRef(registry_model=QetModelRegistry.foundational_matq)
settings = DatasetGenerationWorkflowSettings(
    number_of_initial_structures=NUMBER_OF_INITIAL_STRUCTURES,
    mlip_relax_structures_per_job=4,
    initial_structure_generation_maker_settings=InitialStructureGenerationMakerSettings(
        simulation_settings=InitialStructureGenerationSettings(
            random_seed=42,
            space_groups=[1],
            formula_factory_settings=FixedSetFormulaFactorySettings(
                formula_space=["LiNbO3", "SiO2"],
                random_seed=42,
            ),
        ),
    ),
    mlip_relax_maker_settings=MlipRelaxMakerSettings(
        simulation_settings=MlipRelaxSimulationSettings(
            model=model,
            max_steps=MAX_RELAX_STEPS,
        ),
    ),
    structure_augmentation_and_filtering_maker_settings=StructureAugmentationAndFilteringMakerSettings(
        simulation_settings=StructureAugmentationAndFilteringSettings(
            number_of_rattled_variants=1,
            number_of_stretched_variants=1,
            number_of_stretched_variants_after_rattle=1,
            rattle_settings=RattleAtomPositionsSettings(
                random_seed=42,
                noise_scaling_factor=0.02,
            ),
            stretch_after_rattle_settings=StretchCellSettings(random_seed=42),
            stretch_after_mlip_relax_settings=StretchCellSettings(random_seed=42),
        ),
    ),
    direct_sampling_maker_settings=DirectSamplingMakerSettings(
        simulation_settings=DirectSamplingSettings(
            structure_encoder_model=model,
            birch_clustering_settings=BirchClusteringSettings(number_of_clusters=1),
            cluster_sampling_settings=ClusterSamplingSettings(
                num_samples_per_cluster=4,
            ),
        ),
    ),
)


## 3. Run the dataset generation workflow

This step builds the workflow from the settings and creates an in-memory store for the results and relaxation records. 

Run the workflow locally. Give each job a separate directory for its calculation files.

Wait for this cell to finish before you retrieve the results. The results remain available through `store` while this kernel is active. A kernel restart removes the in-memory results.

In [ ]:
flow = DatasetGenerationWorkflowMaker(simulation_settings=settings).make()
store = JobStore(
    MemoryStore(),
    additional_stores={
        STRUCTURE_BLOB_STORE_NAME: MemoryStore(),
        MLIP_RELAX_TASK_DOCUMENT_BLOB_STORE_NAME: MemoryStore(),
    },
)
RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)

# To use FireWorks, first configure a JobManager named manager.
# Replace the local run below with manager.add_to_launchpad(job_or_flow=flow).
# After the jobs finish, retrieve results from manager.jobstore, not store.
run_locally(
    flow,
    store=store,
    root_dir=RUN_DIRECTORY,
    create_folders=True,
    ensure_success=True,
)


## 4. Retrieve and inspect the selected structures

This step resolves the workflow output from the in-memory store, checks that the selected structures are available. You can run this cell again without repeating the calculations.

**Labelling can be part of this process.** Set `DatasetGenerationWorkflowSettings.structure_labelling_workflow_settings` to include DFT labelling in the generation workflow. This notebook leaves that setting unset. [Notebook 02](02_generate_labels_for_finetuning.ipynb) shows labelling as a separate step, so you can inspect generation and labelling separately. The structures below do not yet have DFT labels.

In [ ]:
result = DatasetGenerationWorkflowTaskDocument.model_validate(
    flow.output.resolve(store)
)
if result.properties is None or not result.properties.structures:
    msg = f"No selected structures. Errors: {result.errors}; warnings: {result.warnings}"
    raise RuntimeError(msg)

structures = result.properties.structures
if result.warnings:
    print(f"Workflow warnings: {result.warnings}")
print(f"Selected structures: {len(structures)}")
for index, structure in enumerate(structures, start=1):
    print(f"\nStructure {index}")
    print(structure)
